# **DSA4060XA Recommender Systems Project**

## Week 1-2: Yelp Restaurant Dataset Preparation
**Project:** TasteMatch - Restaurant Recommender System

**Dataset:** Yelp Open Dataset (https://business.yelp.com/data/resources/open-dataset/)

This notebook:
1. Mounts Google Drive and extracts the Yelp archive
2. Filters businesses to **restaurants**
3. Loads restaurant **reviews in chunks** (memory-safe on Colab free tier)
4. Focuses on one or two cities for a dense rating matrix
5. Saves compact `.parquet` files for reuse
6. Displays dataset heads + summary stats

### 1. Setup - mount Drive and extract the archive
Upload the downloaded Yelp archive (`.tar`) to your Google Drive first.
Adjust `ARCHIVE_PATH` to match where you put it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ARCHIVE_PATH = '/content/drive/MyDrive/Yelp-JSON.zip'
EXTRACT_DIR  = '/content/yelp'

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ARCHIVE_PATH) as z:
    z.extractall(EXTRACT_DIR)

for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        print(os.path.join(root, f))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/yelp/Yelp JSON/Yelp Dataset Documentation & ToS copy.pdf
/content/yelp/Yelp JSON/yelp_dataset.tar
/content/yelp/__MACOSX/Yelp JSON/._yelp_dataset.tar
/content/yelp/__MACOSX/Yelp JSON/._Yelp Dataset Documentation & ToS copy.pdf


### 2. Load businesses and filter to restaurants

In [ ]:
import pandas as pd

BIZ = next(os.path.join(r, f) for r, d, fs in os.walk(EXTRACT_DIR) for f in fs if 'business' in f)
biz = pd.read_json(BIZ, lines=True)

restaurants = biz[biz['categories'].str.contains('Restaurants|Food', case=False, na=False)].copy()
print(f'All businesses: {len(biz):,}')
print(f'Restaurants/food businesses: {len(restaurants):,}')
restaurants.head()

All businesses: 150,346
Restaurants/food businesses: 64,616


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
import tarfile

with tarfile.open('/content/yelp/Yelp JSON/yelp_dataset.tar') as tar:
    tar.extractall('/content/yelp')

for root, dirs, files in os.walk('/content/yelp'):
    for f in files:
        print(os.path.join(root, f))

/tmp/ipykernel_1240/898263658.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/yelp')


/content/yelp/yelp_academic_dataset_review.json
/content/yelp/yelp_academic_dataset_business.json
/content/yelp/yelp_academic_dataset_tip.json
/content/yelp/yelp_academic_dataset_checkin.json
/content/yelp/Dataset_User_Agreement.pdf
/content/yelp/yelp_academic_dataset_user.json
/content/yelp/Yelp JSON/Yelp Dataset Documentation & ToS copy.pdf
/content/yelp/Yelp JSON/yelp_dataset.tar
/content/yelp/__MACOSX/Yelp JSON/._yelp_dataset.tar
/content/yelp/__MACOSX/Yelp JSON/._Yelp Dataset Documentation & ToS copy.pdf


### 3. Choose your city / cities

In [ ]:
restaurants['city'].value_counts().head(15)

,count
city,
Philadelphia,7076
Tampa,3666
Indianapolis,3483
Tucson,3109
Nashville,3051
New Orleans,2808
Edmonton,2666
Saint Louis,2107
Reno,1723


In [ ]:
CITIES = ['Philadelphia']

restaurants = restaurants[restaurants['city'].isin(CITIES)].copy()
rest_ids = set(restaurants['business_id'])
print(f'Restaurants in {CITIES}: {len(restaurants):,}')
restaurants[['name','city','stars','review_count','categories']].head()

Restaurants in ['Philadelphia']: 7,076


,name,city,stars,review_count,categories
3,St Honore Pastries,Philadelphia,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
15,Tuna Bar,Philadelphia,4.0,245,"Sushi Bars, Restaurants, Japanese"
19,BAP,Philadelphia,4.5,205,"Korean, Restaurants"
28,Bar One,Philadelphia,4.0,65,"Cocktail Bars, Bars, Italian, Nightlife, Resta..."
31,DeSandro on Main,Philadelphia,3.0,41,"Pizza, Restaurants, Salad, Soup"


### 4. Load reviews in chunks, keep only restaurant reviews
This is the memory-safe step: never load all ~7M reviews at once.

In [ ]:
REVIEW = f'{EXTRACT_DIR}/yelp_academic_dataset_review.json'

# Keep only the columns needed for modelling (dropping 'text' saves a LOT of memory)
KEEP_COLS = ['review_id', 'user_id', 'business_id', 'stars', 'date', 'useful', 'funny', 'cool']

chunks = pd.read_json(REVIEW, lines=True, chunksize=500_000)
parts = []
for i, chunk in enumerate(chunks):
    parts.append(chunk.loc[chunk['business_id'].isin(rest_ids), KEEP_COLS])
    if (i + 1) % 4 == 0:
        print(f'  processed {(i+1)*500_000:,} reviews...')

reviews = pd.concat(parts, ignore_index=True)
reviews['date'] = pd.to_datetime(reviews['date'])
print(f'Restaurant reviews kept: {len(reviews):,}')
reviews.head()   # SCREENSHOT 3: review/ratings table structure

  processed 2,000,000 reviews...
  processed 4,000,000 reviews...
  processed 6,000,000 reviews...
Restaurant reviews kept: 738,688


,review_id,user_id,business_id,stars,date,useful,funny,cool
0,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,2015-01-04 00:01:03,1,0,1
1,JrIxlS1TzJ-iCu79ul40cQ,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,2015-09-23 23:10:31,1,2,1
2,8JFGBuHMoiNDyfcxuWNtrA,smOvOajNG0lS4Pq7d8g4JQ,RZtGWDLCAtuipwaZ-UfjmQ,4,2009-10-14 19:57:14,0,0,0
3,oyaMhzBSwfGgemSGuZCdwQ,Dd1jQj7S-BFGqRbApFzCFw,YtSqYv1Q_pOltsVPSx54SA,5,2013-06-24 11:21:25,0,0,0
4,Xs8Z8lmKkosqW5mw_sVAoA,IQsF3Rc6IgCzjVV9DE8KXg,eFvzHawVJofxSnD7TgbZtg,5,2014-11-12 15:30:27,0,0,0


### 5. Filter to reasonably active users and restaurants
Reduces sparsity so collaborative filtering and matrix factorization work properly later.

In [ ]:
MIN_REST_REVIEWS = 20   # restaurant must have at least this many reviews
MIN_USER_REVIEWS = 5    # user must have rated at least this many restaurants

reviews = reviews[reviews['business_id'].map(reviews['business_id'].value_counts()) >= MIN_REST_REVIEWS]
reviews = reviews[reviews['user_id'].map(reviews['user_id'].value_counts()) >= MIN_USER_REVIEWS]

# Keep only restaurants that survived the review filter
restaurants = restaurants[restaurants['business_id'].isin(reviews['business_id'].unique())].copy()

n_users  = reviews['user_id'].nunique()
n_items  = reviews['business_id'].nunique()
density  = len(reviews) / (n_users * n_items)
print(f'Users: {n_users:,} | Restaurants: {n_items:,} | Ratings: {len(reviews):,}')
print(f'Rating matrix density: {density:.4%}')   # SCREENSHOT 4: summary stats

Users: 27,771 | Restaurants: 4,495 | Ratings: 436,931
Rating matrix density: 0.3500%


### 6. Load user metadata for the active users

In [ ]:
USER = f'{EXTRACT_DIR}/yelp_academic_dataset_user.json'
active_ids = set(reviews['user_id'])

chunks = pd.read_json(USER, lines=True, chunksize=500_000)
users = pd.concat(
    [c.loc[c['user_id'].isin(active_ids),
           ['user_id','review_count','average_stars','yelping_since','fans']]
     for c in chunks],
    ignore_index=True)
print(f'Active users: {len(users):,}')
users.head()

Active users: 27,771


,user_id,review_count,average_stars,yelping_since,fans
0,j14WgRoU_-2ZE1aw1dXrJg,4333,3.74,2009-01-25 04:35:42,3138
1,NIhcRW6DWvk1JQhDhXwgOQ,2288,3.69,2005-12-30 13:47:19,345
2,RDTVzWPoCeGaUujrHIWRBQ,155,4.11,2010-09-10 16:17:27,5
3,IpLRJY4CP3fXtlEd8Y4GFQ,518,2.95,2009-04-11 14:35:46,35
4,qsHZ6_yT870pmm4Oxvw5Og,39,3.85,2011-06-16 01:55:54,5


### 7. Save compact parquet files to Drive
From now on only ever reload these small files - no more re-processing 5 GB.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/DSA4060/data'
os.makedirs(OUT_DIR, exist_ok=True)

reviews.to_parquet(f'{OUT_DIR}/reviews.parquet', index=False)
users.to_parquet(f'{OUT_DIR}/users.parquet', index=False)

# For businesses, flatten a few useful attribute columns for content-based filtering later
attrs = pd.json_normalize(restaurants['attributes']).add_prefix('attr_')
restaurants_flat = pd.concat(
    [restaurants.drop(columns=['attributes','hours']).reset_index(drop=True),
     attrs.reset_index(drop=True)], axis=1)
restaurants_flat.to_parquet(f'{OUT_DIR}/restaurants.parquet', index=False)

!ls -lh {OUT_DIR}

total 19M
-rw------- 1 root root 486K Sep 18 16:45 restaurants.parquet
-rw------- 1 root root  17M Sep 18 16:45 reviews.parquet
-rw------- 1 root root 1.2M Sep 18 16:45 users.parquet
